# Test local chunking b4 moving to Setonix

In [ ]:
from pathlib import Path
import warnings
import dask
from dask.distributed import Client
import xarray as xr

dpird_path= Path("C:/Users/John/OneDrive/Desktop/ICRAR/The_work/ICRAR-Weather-Forcasting/dataset_DPIRD_utc0_clean/DPIRD_final_stations_utc0.nc")
ecmwf_path= Path("C:/Users/John/OneDrive/Desktop/ICRAR/The_work/ICRAR-Weather-Forcasting/dataset_ecmwf_op_clean/2024/02/06.nc")

tmp_drop_path= Path("../.tmp/compressed")

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning
)

### DPIRD dataset

In [20]:
dpird_ds=xr.open_dataset(dpird_path, engine="h5netcdf")
chunked_dpird_ds= dpird_ds.chunk({'time':3289})
chunked_dpird_ds

<xarray.Dataset> Size: 3GB
Dimensions:                       (station: 192, time: 105248)
Coordinates:
  * station                       (station) <U22 17kB 'Floreat Park' ... 'Gut...
    lat                           (station) float64 2kB dask.array<chunksize=(192,), meta=np.ndarray>
    lon                           (station) float64 2kB dask.array<chunksize=(192,), meta=np.ndarray>
    code                          (station) <U5 4kB dask.array<chunksize=(192,), meta=np.ndarray>
  * time                          (time) datetime64[ns] 842kB 2022-01-01 ... ...
Data variables: (12/18)
    airTemperature                (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    apparentAirTemperature        (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    relativeHumidity              (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    dewPoint                      (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    panEvaporation                (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    evapotranspiration_shortCrop  (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    ...                            ...
    frostCondition                (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    heatCondition                 (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    wind_3m_speed                 (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    wind_3m_degN                  (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    wind_10m_speed                (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
    wind_10m_degN                 (station, time) float64 162MB dask.array<chunksize=(192, 3289), meta=np.ndarray>
Attributes:
    time_zone:      UTC+08:00
    time_standard:  local_time
    comment:        All timestamps are local time in UTC+08:00 (Australia/Per...

### ECMWF dataset

In [ ]:
ecmwf_ds=xr.open_dataset(ecmwf_path, engine="h5netcdf")
chunked_ecmwf_ds= ecmwf_ds.chunk({"time":4,"step":25})
chunked_ecmwf_ds

<xarray.Dataset> Size: 4GB
Dimensions:     (time: 14, step: 113, latitude: 111, longitude: 151)
Coordinates:
  * time        (time) datetime64[ns] 112B 2024-02-06 ... 2024-02-12T12:00:00
  * step        (step) float64 904B 0.0 1.0 2.0 3.0 ... 150.0 156.0 162.0 168.0
    valid_time  (time, step) datetime64[ns] 13kB ...
  * latitude    (latitude) float64 888B -26.0 -26.1 -26.2 ... -36.8 -36.9 -37.0
  * longitude   (longitude) float64 1kB 111.0 111.1 111.2 ... 125.8 125.9 126.0
Data variables: (12/37)
    t2m         (time, step, latitude, longitude) float32 106MB ...
    d2m         (time, step, latitude, longitude) float32 106MB ...
    msl         (time, step, latitude, longitude) float32 106MB ...
    tcc         (time, step, latitude, longitude) float32 106MB ...
    lcc         (time, step, latitude, longitude) float32 106MB ...
    i10fg       (time, step, latitude, longitude) float32 106MB ...
    ...          ...
    u850        (time, step, latitude, longitude) float32 106MB ...

## Chunking & Compression
Compression step is in encoding

In [4]:
client= Client(n_workers=4, threads_per_worker=1)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 13.86 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43794,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:25540,Total threads: 1
Dashboard: http://127.0.0.1:25545/status,Memory: 3.46 GiB
Nanny: tcp://127.0.0.1:43797,


In [ ]:
out_dir = tmp_drop_path
out_dir.mkdir(parents=True, exist_ok=True)

dpird_out = out_dir / "DPIRD_final_stations_utc0.chunked_zlib.nc"
ecmwf_out = out_dir / "2024_02_06.chunked_zlib.nc"

def build_var_encoding(ds, chunk_dict, complevel=5):
    enc = {}
    for v in ds.data_vars:
        var_chunks = tuple(chunk_dict.get(dim, ds[v].sizes[dim]) for dim in ds[v].dims)

        enc[v] = {
            "zlib": True,
            "complevel": complevel,
            "shuffle": True,
            "chunksizes": var_chunks
        }
    return enc

dpird_chunks = {'time': 3289}
ecmwf_chunks = {"time": 4, "step": 25}
                
dpird_encoding = build_var_encoding(dpird_ds, dpird_chunks)
ecmwf_encoding = build_var_encoding(ecmwf_ds, ecmwf_chunks)

# Dask-lazy netCDF writes
dpird_write = dpird_ds.to_netcdf(
    path=dpird_out,
    engine="h5netcdf",
    format= "NETCDF4",
    encoding=dpird_encoding,
    compute=False,
)

ecmwf_write = ecmwf_ds.to_netcdf(
    path=ecmwf_out,
    engine="h5netcdf",
    format= "NETCDF4",
    encoding=ecmwf_encoding,
    compute=False,
)

# Execute both writes in parallel
dask.compute(dpird_write, ecmwf_write)





(None, None)

| Dataset | Original Size | .to_netcdf() compression size | .to_zarr compression() size
|---|---|---|---|
| DPIRD | 2.7GB | 563MB | - |
| ECMWF | 4.0GB | 1.65GB | - |